# 01 - Market Data Exploration

Explores the raw sample OHLCV data: basic statistics, price/volume trends, and a sanity check for the cleaning steps implemented in `src.data.cleaner`.

Run `python scripts/generate_sample_data.py` from the `ml/` directory first if `data/sample/market_data.csv` does not exist yet.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import matplotlib.pyplot as plt
import pandas as pd

from src.config import SAMPLE_DATA_DIR
from src.data.cleaner import clean_market_data
from src.data.loader import load_market_data_csv

%matplotlib inline

In [ ]:
df = load_market_data_csv(SAMPLE_DATA_DIR / "market_data.csv")
print(df.shape)
df.head()

In [ ]:
df.describe()

## Missing values / duplicate timestamps check

In [ ]:
print("Missing values per column:")
print(df.isna().sum())
print("\nDuplicate timestamps:", df['timestamp'].duplicated().sum())

## Clean the data and inspect outlier flags

In [ ]:
cleaned = clean_market_data(df)
print("Rows flagged as return outliers:", cleaned['is_return_outlier'].sum())
cleaned[cleaned['is_return_outlier']].head()

## Price and volume over time

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
axes[0].plot(cleaned['timestamp'], cleaned['close'])
axes[0].set_title('Close price')
axes[1].plot(cleaned['timestamp'], cleaned['volume'], color='orange')
axes[1].set_title('Volume')
plt.tight_layout()
plt.show()

## Return distribution

In [ ]:
returns = cleaned['close'].pct_change().dropna()
plt.figure(figsize=(8, 4))
plt.hist(returns, bins=60)
plt.title('Hourly return distribution')
plt.xlabel('Return')
plt.show()
print(returns.describe())